In [3]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timezone;


df_telemetry = pd.read_csv("../data/raw/sensors_telemetry.csv")
df_graph = pd.read_csv("../data/raw/network_graph.csv")
df_distributor = pd.read_csv('../data/raw/network_nodes_distributors.csv')
df_technicians = pd.read_csv('../data/raw/network_nodes_technicians.csv')
df_payments = pd.read_csv('../data/raw/payments.csv')


In [11]:
df_payments.head(10)

,payment_id,installation_id,amount_xaf,payment_date,channel,status,operator_code
0,ORMN-68800577-738,1223,12700.0,2025-02-27,orange_money,reversed,OM237
1,MOMO-13724595-652,988,5900.0,2024-08-10,mtn_momo,success,MTN909
2,MOMO-20199785-373,4689,7800.0,2024-05-08,mtn_momo,success,MTN188
3,ORMN-86848517-452,2251,4000.0,2025-09-18,orange_money,success,OM237
4,CASH-19700473-657,242,144100.0,2024-02-17,cash,pending,NaN
5,ORMN-17620359-478,4786,17500.0,2025-01-29,orange_money,success,OM97
6,MOMO-91870714-902,1243,43200.0,2024-05-22,mtn_momo,success,MTN833
7,ORMN-30753151-412,619,5500.0,2024-06-01,orange_money,success,OM-CM
8,MOMO-94592836-578,501,4000.0,2024-02-28,mtn_momo,pending,MTN739
9,MOMO-92653287-273,489,14100.0,2024-10-08,mtn_momo,success,MTN316


In [24]:
df_distributor.head(10)

,id,type,name,region,since
0,DIST-001,distributor,Dist. NGANOU & Frères,Littoral,2020-10-08
1,DIST-002,distributor,Dist. HAMADOU & Frères,Centre,2021-12-15
2,DIST-003,distributor,Dist. OUSMANOU & Frères,Ouest,2021-06-21
3,DIST-004,distributor,Dist. FEUDJIO & Frères,Est,2022-10-10
4,DIST-005,distributor,Dist. EKANI & Frères,Est,2022-06-16
5,DIST-006,distributor,Dist. ATANGANA & Frères,Ouest,2021-10-31
6,DIST-007,distributor,Dist. NGUIMO & Frères,Centre,2021-11-27
7,DIST-008,distributor,Dist. TCHUENKAM & Frères,Ouest,2022-07-26
8,DIST-009,distributor,Dist. OUSMANOU & Frères,Ouest,2020-05-31
9,DIST-010,distributor,Dist. FOTSO & Frères,Centre,2022-10-29


In [10]:
# Parse date normalisation  
import re


def normalisation_date(raw):
    
    if pd.isna(raw) or str(raw).strip() == "":
        return datetime.now()
    
    formats_possibles = [
        "%Y-%m-%d",  # Format YYYY-MM-DD 
        "%d/%m/%Y",  # Format DD/MM/YYYY
        "%d-%m-%Y",  # Format DD-MM-YYYY
        "%Y/%m/%d",  # Format YYYY/MM/DD
        "%m/%d/%Y",  # Format MM/DD/YYYY 
        "%m-%d-%Y",  # Format MM-DD-YYYY 
        "%Y/%d/%m",  # Format YYYY/DD/MM 
        "%Y-%d-%m",  # Format YYYY-DD-MM 
    ]
    
    for fmt in formats_possibles:
        try:
            return datetime.strptime(raw, fmt)
        except ValueError:
            continue
    
    print(f"⚠️  Date non reconnu : '{raw}' → remplacé par now()")
    return datetime.now()

def normalisation_telephone(raw):
    """
    Normalise un numéro de téléphone au format Camerounais : +2376XXXXXXXX
    """
    if pd.isna(raw) or str(raw).strip() == "":
        return "INCONNU"
        
    raw_str = str(raw).strip()
    
    clean = re.sub(r'\D', '', raw_str)
    
    if clean.startswith('0'):
        clean = clean[1:]
        
    if clean.startswith('00237'):
        clean = '237' + clean[5:]
        
    if clean.startswith('6'):
        clean = '237' + clean
        
    if len(clean) == 12 and clean.startswith('237') and clean[3] == '6':
        return '+' + clean
    else:
        print(f"⚠️  Numéro suspect : '{raw_str}' -> formaté en +{clean}")
        return '+' + clean




df_graph['date'] = df_graph['date'].map(normalisation_date) 
df_distributor['since'] = df_distributor['since'].map(normalisation_date) 
df_technicians['phone'] = df_technicians['phone'].map(normalisation_telephone)
df_payments['payment_date'] = df_payments['payment_date'].map(normalisation_date)

⚠️  Numéro suspect : '237-262-402-726' -> formaté en +237262402726
⚠️  Numéro suspect : '+237207167992' -> formaté en +237207167992
⚠️  Numéro suspect : '+237224425095' -> formaté en +237224425095
⚠️  Numéro suspect : '+237288999734' -> formaté en +237288999734
⚠️  Numéro suspect : '235013941' -> formaté en +235013941
⚠️  Numéro suspect : '245763168' -> formaté en +245763168
⚠️  Numéro suspect : '237-257-404-189' -> formaté en +237257404189
⚠️  Numéro suspect : '237-218-613-816' -> formaté en +237218613816
⚠️  Numéro suspect : '220888293' -> formaté en +220888293
⚠️  Numéro suspect : '237-295-055-343' -> formaté en +237295055343
⚠️  Numéro suspect : '237-287-531-007' -> formaté en +237287531007
⚠️  Numéro suspect : '237-235-141-299' -> formaté en +237235141299
⚠️  Numéro suspect : '237-236-371-001' -> formaté en +237236371001
⚠️  Numéro suspect : '237-223-568-955' -> formaté en +237223568955
⚠️  Numéro suspect : '234283124' -> formaté en +234283124
⚠️  Numéro suspect : '269761830' -> f

In [4]:
df_graph['date'] = pd.to_datetime(df_graph['date'],errors='coerce')

In [17]:
df_telemetry.head(5)

,sensor_id,installation_id,timestamp,solar_output_w,battery_level_pct,consumption_w,alert_code,region
0,CAM-4678-S1,4678,2025-01-06 21:45:05+00:00,0.0,32.8,228.68,FAULT_02,Ouest
1,CAM-1187-S1,1187,2025-06-18 05:49:56+00:00,0.0,46.4,67.97,OVERCURRENT,Adamaoua
2,CAM-4076-S1,4076,2025-01-12 23:34:09+00:00,0.0,60.0,26.87,OVERCURRENT,Adamaoua
3,CAM-3602-S1,3602,2025-02-08 03:13:34+00:00,0.0,40.3,43.43,OVR_V,Centre
4,CAM-4490-S1,4490,2025-01-22 06:13:28+00:00,0.0,48.8,265.51,FAULT_02,Littoral


In [3]:
df_telemetry.info()

<class 'pandas.DataFrame'>
RangeIndex: 120960 entries, 0 to 120959
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   sensor_id          120960 non-null  str    
 1   installation_id    120960 non-null  int64  
 2   timestamp          120960 non-null  str    
 3   solar_output_w     68332 non-null   float64
 4   battery_level_pct  120960 non-null  float64
 5   consumption_w      116187 non-null  float64
 6   alert_code         80453 non-null   str    
 7   region             120960 non-null  str    
dtypes: float64(3), int64(1), str(4)
memory usage: 7.4 MB


In [16]:
df_telemetry[["alert_code", "battery_level_pct"]]

,alert_code,battery_level_pct
0,FAULT_02,32.8
1,OVERCURRENT,46.4
2,OVERCURRENT,60.0
3,OVR_V,40.3
4,FAULT_02,48.8
...,...,...
120955,FAULT_01,67.6
120956,OVR_V,84.8
120957,FAULT_01,47.3
120958,NO_SIG,48.8


In [16]:
def parse_timestamp(raw) -> datetime:
    """
    Convertit 3 formats de timestamps en datetime UTC :
    - Unix epoch  : 1738984414
    - ISO 8601    : 2024-01-15T14:30:00
    - DD/MM/YYYY  : 15/01/2024 14:30
    - MM/DD/YYYY
    """
    if pd.isna(raw):
        return datetime.now(timezone.utc)

    raw_str = str(raw).strip()

    # Format Unix epoch (10 chiffres)
    if re.match(r'^\d{10}$', raw_str):
        return datetime.fromtimestamp(int(raw_str), tz=timezone.utc)

    # Format ISO 8601
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%SZ"):
        try:
            return datetime.strptime(raw_str, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            continue

    # Format DD/MM/YYYY HH:MM
    for fmt in ("%d/%m/%Y %H:%M", "%d/%m/%Y %H:%M:%S"):
        try:
            return datetime.strptime(raw_str, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            continue
    
        
        
    # Format 27-04-2025 09:17:13    
    for fmt in ("%d-%m-%Y %H:%M:%S", "%m-%d-%Y %H:%M:%s"):
        try:
            return datetime.strptime(raw_str, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            continue     

    # Fallback : maintenant
    print(f"  ⚠️  Timestamp non reconnu : '{raw_str}' → remplacé par now()")
    return datetime.now(timezone.utc)



def normalisation_region(raw:str) -> str:
    if not raw or pd.isna(raw):
        return 'Inconnu'
    
    raw = str(raw).strip().lower()
    raw = raw.title()
    
    if raw in {'Ouest', 'Oue', 'Ouest-Cm'}:
        return 'Ouest'
    if raw in {'Lit.', 'Ltl'}:
        return 'Littoral'
    if raw in {'Nord_Ouest', 'No'}:
        return 'Nord-Ouest'
    if raw in {'E', 'Est-Cm','Est'}:
        return 'Est'
    if raw in {'Ada', 'Adamawa'}:
        return 'Adamaoua'
    if raw in {'Ctr'}:
        return 'Centre'
    if raw in {'Ext-Nord', 'Extreme-Nord', 'En'}:
        return 'Extrême-Nord'
     
    return raw      
    
    
def normalisation_alert(raw:str)->str:
    if not raw or pd.isna(raw) :
        return "AUCUNE"  
    raw = str(raw).strip().upper()
    return raw


df_telemetry['battery_level_pct'] = df_telemetry['battery_level_pct'].clip(0,100)

df_telemetry["solar_output_w"]    = pd.to_numeric(df_telemetry["solar_output_w"], errors="coerce").fillna(0.0)
df_telemetry["battery_level_pct"] = pd.to_numeric(df_telemetry["battery_level_pct"], errors="coerce").fillna(0.0)
df_telemetry["consumption_w"]     = pd.to_numeric(df_telemetry["consumption_w"], errors="coerce").fillna(0.0)

df_telemetry['alert_code'] = df_telemetry['alert_code'].apply(normalisation_alert)    

df_telemetry['region'] = df_telemetry['region'].map(normalisation_region) 


df_telemetry['timestamp'] = df_telemetry['timestamp'].apply(parse_timestamp)

In [17]:
display(df_telemetry)

,sensor_id,installation_id,timestamp,solar_output_w,battery_level_pct,consumption_w,alert_code,region
0,CAM-4678-S1,4678,2025-01-06 21:45:05+00:00,0.00,32.8,228.68,FAULT_02,Ouest
1,CAM-1187-S1,1187,2025-06-18 05:49:56+00:00,0.00,46.4,67.97,OVERCURRENT,Adamaoua
2,CAM-4076-S1,4076,2025-01-12 23:34:09+00:00,0.00,60.0,26.87,OVERCURRENT,Adamaoua
3,CAM-3602-S1,3602,2025-02-08 03:13:34+00:00,0.00,40.3,43.43,OVR_V,Centre
4,CAM-4490-S1,4490,2025-01-22 06:13:28+00:00,0.00,48.8,265.51,FAULT_02,Littoral
...,...,...,...,...,...,...,...,...
120955,CAM-0048-S1,48,2025-01-02 08:25:29+00:00,1053.39,67.6,245.26,FAULT_01,Centre
120956,CAM-2234-S1,2234,2025-01-27 16:06:27+00:00,1454.19,84.8,77.74,OVR_V,Littoral
120957,CAM-0492-S1,492,2025-03-01 05:31:00+00:00,0.00,47.3,35.58,FAULT_01,Centre
120958,CAM-3814-S1,3814,2025-01-02 06:34:46+00:00,0.01,48.8,289.25,NO_SIG,Nord-Ouest


In [ ]:
2
chemin = "../data/processed/sensors_telemetry.csv"

# La commande d'export
df_telemetry.to_csv(chemin, index=False, encoding='utf-8')

print(f"✅ Fichier sauvegardé avec succès sous : {chemin}")
print(f"📊 Nombre de lignes exportées : {len(df_telemetry)}")

✅ Fichier sauvegardé avec succès sous : ../data/processed/sensors_telemetry.csv
📊 Nombre de lignes exportées : 120960


In [26]:
chemin = "../data/processed/network_graph.csv"

df_graph.to_csv(chemin, index=False, encoding='utf-8')

print(f"Nombre de lignes : {len(df_graph)} save to {chemin}")

Nombre de lignes : 10514 save to ../data/processed/network_graph.csv


In [27]:
chemin = "../data/processed/network_nodes_technicians.csv"

df_technicians.to_csv(chemin, index=False, encoding='utf-8')

print(f"Nombre de lignes : {len(df_technicians)} save to {chemin}")

Nombre de lignes : 210 save to ../data/processed/network_nodes_technicians.csv


In [28]:
chemin = "../data/processed/network_nodes_distributors.csv"

df_distributor.to_csv(chemin, index=False, encoding='utf-8')

print(f"Nombre de lignes : {len(df_distributor)} save to {chemin}")

Nombre de lignes : 38 save to ../data/processed/network_nodes_distributors.csv


In [12]:
chemin = "../data/processed/payments.csv"

df_payments.to_csv(chemin, index=False, encoding='utf-8')

print(f"Nombre de lignes : {len(df_payments)} save to {chemin}")

Nombre de lignes : 28140 save to ../data/processed/payments.csv


In [4]:
# On crée une fausse clé primaire comme Cassandra le ferait
df_telemetry['fake_pk'] = df_telemetry['region'] + str(df_telemetry['timestamp']) + df_telemetry['sensor_id']

# On compte combien de doublons il y a
nombre_doublons = df_telemetry.duplicated(subset=['fake_pk']).sum()
print(f"Nombre de lignes qui vont s'écraser dans Cassandra : {nombre_doublons}")

Nombre de lignes qui vont s'écraser dans Cassandra : 115675
